# PauseBeforePay · Backend Prototype (Colab)
# PauseBeforePay · 后端原型（Colab 版）

PE6201 end-of-course project. This notebook contains:

PE6201 期末项目。本笔记本包含：

1. **Calculator** — deterministic Python calculations for three options: pay in full, instalment, delay<br>
   **计算器**：全款 / 分期 / 延迟购买三种方案，全部用 Python 确定性计算
2. **Calculation trace** — every step prints its formula, the numbers substituted, and the result<br>
   **运算过程**：每一步的公式、代入的数字和结果都会打印出来
3. **L1 tests** — the calculator checked against hand-calculated results<br>
   **L1 测试**：用手算结果核对计算器
4. **Haiku explanation module** — turns results into plain language and automatically checks every number the model writes<br>
   **Haiku 解释模块**：把计算结果翻译成大白话，并自动校验模型写出的每个数字
5. **Cost logging** — tokens and USD cost for each call<br>
   **成本记录**：每次调用的 token 数和美元成本

**One-time setup:** click the key icon (Secrets) in the left sidebar → add `ANTHROPIC_API_KEY` with your API key → turn on "Notebook access". Parts 1–3 run without an API key.

**使用前准备（只需一次）**：点左侧栏钥匙图标（Secrets）→ 新建 `ANTHROPIC_API_KEY` 并填入 API key → 打开 "Notebook access"。第 1–3 部分不需要 API key。

In [1]:
!pip install -q anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 11.9 MB/s eta 0:00:00


## Part 1 · Calculator ｜ 第 1 部分 · 计算器

Core formulas ｜ 核心公式：

| Quantity 量 | Formula 公式 |
|---|---|
| Monthly fixed outflow 每月固定支出 | essentials + existing debt payments 必要支出 + 已有月供 |
| Monthly surplus 每月结余 | income − fixed outflow 收入 − 每月固定支出 |
| Emergency runway 应急金可撑月数 | savings ÷ fixed outflow 储蓄 ÷ 每月固定支出 |
| Instalment, flat fee 分期月供（按期手续费） | price ÷ n + price × fee rate 价格 ÷ 期数 + 价格 × 每期费率 |
| Instalment, amortised 分期月供（等额本息） | P × i ÷ (1 − (1+i)^−n), i = annual rate ÷ 12 |
| Total repaid 分期总还款 | instalment × n + upfront fee 月供 × 期数 + 一次性手续费 |
| Effective annual rate 实际年化利率 | solve r: amount received = Σ instalment ÷ (1+r)^t; EAR = (1+r)^12 − 1 |
| Debt-to-income 月还款收入比 | (existing debt + instalment) ÷ income (已有月供 + 分期月供) ÷ 收入 |
| Months to save (delay) 延迟购买需要几个月 | ceil(price ÷ monthly saving) 向上取整(价格 ÷ 每月可存金额) |

Risk levels also come from fixed rules, not from the model: insufficient savings, negative monthly cash flow, runway below target, debt-to-income above the warning line, and extra borrowing cost. Thresholds can be adjusted in `Settings`.

风险判断同样是固定规则，不交给模型：储蓄不够、每月现金流为负、应急金低于目标、月还款收入比超过警戒线、有额外借贷成本。阈值可在 `Settings` 中调整。

In [38]:
import math
from dataclasses import dataclass, asdict
from typing import Optional


@dataclass
class UserProfile:
    monthly_income: float            # monthly take-home income / 每月税后收入
    cash_savings: float              # current cash savings / 现有现金储蓄
    essential_expenses: float        # monthly essentials: rent, food, transport / 每月必要支出（房租、饭钱、交通等）
    existing_debt_payments: float = 0.0          # existing monthly debt repayments / 每月已有的还款
    monthly_saving_capacity: Optional[float] = None  # monthly saving for the delay option; defaults to monthly surplus / 延迟购买时每月能存多少，不填则默认为每月结余


@dataclass
class Purchase:
    price: float                     # purchase price / 商品价格
    instalment_months: int           # number of instalments / 分期期数
    monthly_fee_rate: float = 0.0    # flat fee per instalment on the original price, e.g. 0.006 = 0.6% / 按原价每期收取的手续费率
    annual_interest_rate: float = 0.0  # amortised annual interest rate, e.g. 0.12 = 12% / 等额本息年利率
    upfront_fee: float = 0.0         # one-time upfront fee / 一次性手续费


@dataclass
class Settings:
    emergency_target_months: float = 3.0   # target emergency runway in months / 应急金目标月数
    max_debt_to_income: float = 0.35       # debt-to-income warning line; adjustable, not a regulatory standard / 月还款收入比警戒线（可调，非监管标准）
    severe_debt_to_income: float = 0.55    # severe line, based on MAS TDSR cap for property loans / 严重线，参考 MAS 房贷 TDSR 上限
    currency: str = "SGD"


# ---------------------------------------------------------------- Helpers / 工具函数
class StepLog:
    """Records every calculation step (label, formula, numbers, result) for display.
    记录每一步运算：名称、公式、代入数字、结果，前端可直接展示。"""
    def __init__(self):
        self.steps = []

    def add(self, option, label, formula, substitution, result):
        self.steps.append({"option": option, "label": label, "formula": formula,
                           "substitution": substitution, "result": result})
        return result


def money(x):
    return f"{x:,.2f}"


def pct(x):
    return f"{x * 100:.2f}%"


def r2(x):
    return None if x is None else round(x, 2)


def r4(x):
    return None if x is None else round(x, 4)


# ---------------------------------------------------------------- Input validation / 输入校验
def validate_inputs(p: UserProfile, q: Purchase):
    errors = []
    if p.monthly_income <= 0:
        errors.append("monthly_income must be greater than 0.")
    for name in ["cash_savings", "essential_expenses", "existing_debt_payments"]:
        if getattr(p, name) is None or getattr(p, name) < 0:
            errors.append(f"{name} must be 0 or greater.")
    if p.monthly_saving_capacity is not None and p.monthly_saving_capacity < 0:
        errors.append("monthly_saving_capacity must be 0 or greater.")
    if q.price <= 0:
        errors.append("price must be greater than 0.")
    if not isinstance(q.instalment_months, int) or q.instalment_months < 1:
        errors.append("instalment_months must be a whole number of at least 1.")
    if q.monthly_fee_rate < 0 or q.annual_interest_rate < 0 or q.upfront_fee < 0:
        errors.append("Fees and interest rates cannot be negative.")
    if q.monthly_fee_rate > 0 and q.annual_interest_rate > 0:
        errors.append("Provide either monthly_fee_rate or annual_interest_rate, not both.")
    return errors


# ---------------------------------------------------------------- Effective annual rate (IRR) / 实际年化利率
def effective_monthly_rate(amount_received, payment, n):
    """Solve monthly rate r so that amount_received = Σ payment / (1+r)^t, t = 1..n (bisection).
    用二分法求月利率 r。"""
    if payment * n <= amount_received + 1e-9:
        return 0.0

    def pv(r):
        return sum(payment / (1 + r) ** t for t in range(1, n + 1))

    lo, hi = 0.0, 1.0
    for _ in range(200):
        mid = (lo + hi) / 2
        if pv(mid) > amount_received:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2


# ---------------------------------------------------------------- Main simulation / 主计算
def simulate(p: UserProfile, q: Purchase, s: Settings = Settings()):
    errors = validate_inputs(p, q)
    if errors:
        return {"ok": False, "errors": errors}

    log = StepLog()
    c = s.currency
    n = q.instalment_months

    # ---- Baseline / 基础量
    base_outflow = p.essential_expenses + p.existing_debt_payments
    log.add("baseline", "Monthly fixed outflow",
            "essential expenses + existing debt payments",
            f"{money(p.essential_expenses)} + {money(p.existing_debt_payments)}",
            f"{c} {money(base_outflow)}")

    disposable = p.monthly_income - base_outflow
    log.add("baseline", "Monthly surplus before purchase",
            "income - monthly fixed outflow",
            f"{money(p.monthly_income)} - {money(base_outflow)}",
            f"{c} {money(disposable)}")

    runway_now = p.cash_savings / base_outflow if base_outflow > 0 else None
    log.add("baseline", "Emergency runway today",
            "cash savings / monthly fixed outflow",
            f"{money(p.cash_savings)} / {money(base_outflow)}",
            f"{runway_now:.2f} months" if runway_now is not None else "unlimited (no fixed outflow)")

    # ---- Option A: pay in full / 方案 A：全款
    savings_after_full = p.cash_savings - q.price
    log.add("pay_in_full", "Savings left after paying in full",
            "cash savings - price",
            f"{money(p.cash_savings)} - {money(q.price)}",
            f"{c} {money(savings_after_full)}")
    runway_full = (max(savings_after_full, 0) / base_outflow) if base_outflow > 0 else None
    log.add("pay_in_full", "Emergency runway after paying in full",
            "max(savings left, 0) / monthly fixed outflow",
            f"{money(max(savings_after_full, 0))} / {money(base_outflow)}",
            f"{runway_full:.2f} months" if runway_full is not None else "unlimited")

    full = {
        "feasible": savings_after_full >= 0,
        "total_paid": r2(q.price),
        "cost_of_credit": 0.0,
        "monthly_payment": 0.0,
        "savings_after_purchase": r2(savings_after_full),
        "monthly_surplus_after": r2(disposable),
        "emergency_runway_months": r2(runway_full),
        "debt_to_income": r4(p.existing_debt_payments / p.monthly_income),
    }

    # ---- Option B: instalment / 方案 B：分期
    P = q.price
    if q.annual_interest_rate > 0:
        i = q.annual_interest_rate / 12
        pmt = P * i / (1 - (1 + i) ** -n)
        log.add("instalment", "Monthly instalment (amortised interest)",
                "P × i / (1 - (1 + i)^-n), i = annual rate / 12",
                f"{money(P)} × {i:.6f} / (1 - (1 + {i:.6f})^-{n})",
                f"{c} {money(pmt)}")
    else:
        principal_part = P / n
        fee_part = P * q.monthly_fee_rate
        pmt = principal_part + fee_part
        log.add("instalment", "Monthly instalment (flat fee on original price)",
                "price / n + price × monthly fee rate",
                f"{money(P)} / {n} + {money(P)} × {q.monthly_fee_rate}",
                f"{money(principal_part)} + {money(fee_part)} = {c} {money(pmt)}")

    total_inst = pmt * n + q.upfront_fee
    log.add("instalment", "Total repaid",
            "monthly instalment × n + upfront fee",
            f"{money(pmt)} × {n} + {money(q.upfront_fee)}",
            f"{c} {money(total_inst)}")

    cost_credit = total_inst - P
    log.add("instalment", "Extra cost of borrowing",
            "total repaid - price",
            f"{money(total_inst)} - {money(P)}",
            f"{c} {money(cost_credit)}")

    r_m = effective_monthly_rate(P - q.upfront_fee, pmt, n)
    ear = (1 + r_m) ** 12 - 1
    log.add("instalment", "Effective annual rate (what the fees really cost)",
            "solve r: price - upfront fee = Σ instalment / (1+r)^t; EAR = (1+r)^12 - 1",
            f"{money(P - q.upfront_fee)} = Σ {money(pmt)} / (1+r)^t, t = 1..{n}",
            f"monthly r = {pct(r_m)}, EAR = {pct(ear)}")

    surplus_inst = disposable - pmt
    log.add("instalment", "Monthly surplus during instalment",
            "monthly surplus before purchase - monthly instalment",
            f"{money(disposable)} - {money(pmt)}",
            f"{c} {money(surplus_inst)}")

    dti_inst = (p.existing_debt_payments + pmt) / p.monthly_income
    log.add("instalment", "Debt-to-income ratio",
            "(existing debt payments + instalment) / income",
            f"({money(p.existing_debt_payments)} + {money(pmt)}) / {money(p.monthly_income)}",
            pct(dti_inst))

    savings_after_inst = p.cash_savings - q.upfront_fee
    outflow_inst = base_outflow + pmt
    runway_inst = savings_after_inst / outflow_inst if outflow_inst > 0 else None
    log.add("instalment", "Emergency runway during instalment",
            "(cash savings - upfront fee) / (monthly fixed outflow + instalment)",
            f"{money(savings_after_inst)} / {money(outflow_inst)}",
            f"{runway_inst:.2f} months" if runway_inst is not None else "unlimited")

    inst = {
        "feasible": surplus_inst >= 0,
        "total_paid": r2(total_inst),
        "cost_of_credit": r2(cost_credit),
        "monthly_payment": r2(pmt),
        "monthly_fee_amount": r2(P * q.monthly_fee_rate),   # fee part of each instalment ｜ 每期月供中的手续费部分
        "instalment_months": n,
        "effective_annual_rate": r4(ear),
        "savings_after_purchase": r2(savings_after_inst),
        "monthly_surplus_after": r2(surplus_inst),
        "emergency_runway_months": r2(runway_inst),
        "debt_to_income": r4(dti_inst),
    }

    # ---- Option C: delay; assumes saving only from monthly surplus, savings untouched
    #      方案 C：延迟购买（假设只用每月结余攒钱，现有储蓄不动）
    capacity = p.monthly_saving_capacity if p.monthly_saving_capacity is not None else max(disposable, 0)
    if capacity > 0:
        months_wait = math.ceil(P / capacity)
        log.add("delay", "Months needed to save up",
                "ceil(price / monthly saving capacity)",
                f"ceil({money(P)} / {money(capacity)})",
                f"{months_wait} months")
    else:
        months_wait = None
        log.add("delay", "Months needed to save up",
                "ceil(price / monthly saving capacity)",
                f"monthly saving capacity = {money(capacity)}",
                "not reachable: no monthly surplus to save")

    delay = {
        "feasible": months_wait is not None,
        "total_paid": r2(P),
        "cost_of_credit": 0.0,
        "monthly_payment": 0.0,
        "monthly_saving_capacity": r2(capacity),
        "months_to_save": months_wait,
        "savings_after_purchase": r2(p.cash_savings),
        "monthly_surplus_after": r2(disposable),
        "emergency_runway_months": r2(runway_now),
        "debt_to_income": r4(p.existing_debt_payments / p.monthly_income),
    }

    options = {"pay_in_full": full, "instalment": inst, "delay": delay}
    for name, o in options.items():
        o["risk_flags"], o["risk_level"] = assess_risk(name, o, s)

    return {
        "ok": True,
        "currency": c,
        "inputs": {**asdict(p), **asdict(q)},
        "settings": asdict(s),
        "baseline": {
            "monthly_fixed_outflow": r2(base_outflow),
            "monthly_surplus": r2(disposable),
            "emergency_runway_months": r2(runway_now),
        },
        "options": options,
        "steps": log.steps,
        "assumptions": [
            "Delay option saves only from the monthly surplus and leaves current savings untouched.",
            "Emergency runway = savings divided by fixed monthly outflow (essentials + debt payments).",
            "No late fees, missed payments or income changes are modelled.",
        ],
    }


# ---------------------------------------------------------------- Risk rules: deterministic, not the model / 风险规则（确定性，不靠模型）
def assess_risk(name, o, s: Settings):
    flags = []
    severe = {"insufficient_savings", "negative_monthly_cash_flow", "cannot_save_up", "severe_debt_to_income"} # Moved initialization here

    if name == "pay_in_full" and o["savings_after_purchase"] < 0:
        flags.append("insufficient_savings")
    if o["monthly_surplus_after"] is not None and o["monthly_surplus_after"] < 0:
        flags.append("negative_monthly_cash_flow")
    runway = o["emergency_runway_months"]
    if runway is not None and runway < s.emergency_target_months:
        flags.append("below_emergency_target")
    if o["debt_to_income"] > s.severe_debt_to_income:
        flags.append("severe_debt_to_income")
    elif o["debt_to_income"] > s.max_debt_to_income:
        flags.append("high_debt_to_income")
    if o["cost_of_credit"] > 0:
        flags.append("extra_borrowing_cost")
    if name == "delay" and o["months_to_save"] is None:
        flags.append("cannot_save_up")

    if severe & set(flags) or (runway is not None and runway < 1):
        level = "high"
    elif {"below_emergency_target", "high_debt_to_income"} & set(flags):
        level = "medium"
    else:
        level = "low"
    return flags, level

## Part 2 · Calculation Trace ｜ 第 2 部分 · 查看运算过程

Change the numbers below to try other cases. Example: monthly income 3,500, savings 5,000, buying a 6,000 laptop in 12 instalments with a 0.6% fee per instalment.

改下面的数字即可尝试不同情况。示例：月收入 3,500，储蓄 5,000，分 12 期买 6,000 的电脑，每期手续费 0.6%。

Look at the effective annual rate step. A 0.6% fee per month looks like only 7.2% a year, but because the fee is always charged on the **original price** while the amount owed falls every month, the true annual cost is almost double. This is exactly how low monthly payments mislead users.

注意"实际年化利率"这一步：每期 0.6% 看起来一年只有 7.2%，但手续费始终按**原价**收取，而欠款每月在减少，所以真实年化成本接近两倍。这正是低月供误导用户的地方。

In [10]:
def print_steps(calc):
    if not calc["ok"]:
        print("Input errors:")
        for e in calc["errors"]:
            print("  -", e)
        return
    current = None
    for s in calc["steps"]:
        if s["option"] != current:
            current = s["option"]
            print(f"\n===== {current.upper()} =====")
        print(f"{s['label']}")
        print(f"   formula : {s['formula']}")
        print(f"   numbers : {s['substitution']}")
        print(f"   result  : {s['result']}")


def print_summary(calc):
    rows = ["total_paid", "cost_of_credit", "monthly_payment", "savings_after_purchase",
            "monthly_surplus_after", "emergency_runway_months", "debt_to_income", "risk_level"]
    opts = calc["options"]
    print(f"\n{'':28}" + "".join(f"{k:>16}" for k in opts))
    for r in rows:
        print(f"{r:28}" + "".join(f"{str(opts[k].get(r)):>16}" for k in opts))
    print(f"{'risk_flags':28}")
    for k, o in opts.items():
        print(f"   {k:14}: {', '.join(o['risk_flags']) or 'none'}")


profile = UserProfile(monthly_income=3500, cash_savings=5000,
                      essential_expenses=2200, existing_debt_payments=300)
purchase = Purchase(price=6000, instalment_months=12, monthly_fee_rate=0.006)
settings = Settings(currency="SGD")

calc = simulate(profile, purchase, settings)
print_steps(calc)
print_summary(calc)


===== BASELINE =====
Monthly fixed outflow
   formula : essential expenses + existing debt payments
   numbers : 2,200.00 + 300.00
   result  : SGD 2,500.00
Monthly surplus before purchase
   formula : income - monthly fixed outflow
   numbers : 3,500.00 - 2,500.00
   result  : SGD 1,000.00
Emergency runway today
   formula : cash savings / monthly fixed outflow
   numbers : 5,000.00 / 2,500.00
   result  : 2.00 months

===== PAY_IN_FULL =====
Savings left after paying in full
   formula : cash savings - price
   numbers : 5,000.00 - 6,000.00
   result  : SGD -1,000.00
Emergency runway after paying in full
   formula : max(savings left, 0) / monthly fixed outflow
   numbers : 0.00 / 2,500.00
   result  : 0.00 months

===== INSTALMENT =====
Monthly instalment (flat fee on original price)
   formula : price / n + price × monthly fee rate
   numbers : 6,000.00 / 12 + 6,000.00 × 0.006
   result  : 500.00 + 36.00 = SGD 536.00
Total repaid
   formula : monthly instalment × n + upfront fee
 

## Part 3 · L1 Tests (Deterministic Assertions) ｜ 第 3 部分 · L1 测试（确定性断言）

Run after every code change. Expected values are hand-calculated or standard reference values; if everything passes, you will see `All L1 tests passed`. This section can go straight into the evaluation chapter of the report.

每次改代码后运行。预期值均为手算或公认标准值，全部通过会显示 `All L1 tests passed`。这部分可直接写进报告的评估章节。

In [11]:
def approx(a, b, tol=0.01):
    return abs(a - b) <= tol

# 1. 0% instalment: payment = price / n, no extra cost ｜ 0% 分期：月供 = 价格 / 期数，无额外成本
c = simulate(UserProfile(2000, 3000, 1400), Purchase(1800, 12))
i = c["options"]["instalment"]
assert approx(i["monthly_payment"], 150) and approx(i["total_paid"], 1800) and i["effective_annual_rate"] == 0

# 2. Flat fee: 6000 × 0.6% × 12 = 432 extra ｜ 按期手续费：额外成本 432
c = simulate(UserProfile(3500, 5000, 2200, 300), Purchase(6000, 12, monthly_fee_rate=0.006))
i = c["options"]["instalment"]
assert approx(i["monthly_payment"], 536) and approx(i["total_paid"], 6432) and approx(i["cost_of_credit"], 432)
assert i["effective_annual_rate"] > 0.072 * 1.8          # true annual cost far above the headline 7.2% ｜ 真实年化远高于表面 7.2%
assert c["options"]["pay_in_full"]["risk_level"] == "high"  # savings cannot cover full payment ｜ 储蓄不够全款

# 3. Amortised reference: 12,000 at 12% p.a. over 12 months -> 1,066.19 ｜ 等额本息标准值
c = simulate(UserProfile(5000, 10000, 2000), Purchase(12000, 12, annual_interest_rate=0.12))
assert approx(c["options"]["instalment"]["monthly_payment"], 1066.19)

# 4. Delay: surplus 1,000/month, price 6,000 -> 6 months ｜ 延迟购买需 6 个月
c = simulate(UserProfile(3500, 5000, 2200, 300), Purchase(6000, 12))
assert c["options"]["delay"]["months_to_save"] == 6

# 5. Edge case: zero savings ｜ 边界：零储蓄
c = simulate(UserProfile(2500, 0, 2000), Purchase(1200, 6))
assert c["ok"] and all(o["risk_level"] == "high" for o in c["options"].values())

# 6. Edge case: no surplus, delay not possible ｜ 边界：无结余，无法延迟购买
c = simulate(UserProfile(2000, 1000, 2000), Purchase(500, 3))
assert c["options"]["delay"]["months_to_save"] is None and "cannot_save_up" in c["options"]["delay"]["risk_flags"]

# 7. Adversarial: contradictory/invalid input must be rejected ｜ 对抗：矛盾或非法输入必须被拦下
c = simulate(UserProfile(0, -1, 100), Purchase(100, 0, monthly_fee_rate=0.01, annual_interest_rate=0.1))
assert not c["ok"] and len(c["errors"]) == 4
# 8. Two-tier debt-to-income: 40% -> medium, 60% -> high ｜ 月还款收入比两档：40% 为中风险，60% 为高风险
c = simulate(UserProfile(2000, 20000, 500), Purchase(8000, 10))
assert "high_debt_to_income" in c["options"]["instalment"]["risk_flags"] and c["options"]["instalment"]["risk_level"] == "medium"
c = simulate(UserProfile(2000, 20000, 500), Purchase(12000, 10))
assert "severe_debt_to_income" in c["options"]["instalment"]["risk_flags"] and c["options"]["instalment"]["risk_level"] == "high"
print("All L1 tests passed")

All L1 tests passed


## Part 4 · Haiku Explanation + Number Validation ｜ 第 4 部分 · Haiku 解释模块 + 数字校验

The model receives only the calculation results (JSON). Its job is to explain, not to calculate. Each response then goes through three checks:

模型只拿到计算结果（JSON），任务是解释而不是计算。拿回结果后做三道检查：

1. Is it valid JSON? ｜ 是否为合法 JSON
2. Are all fields present (explanation and main risk for all three options)? ｜ 字段是否齐全（三个方案都有解释和主要风险）
3. **Number check** — every number the model writes is compared with the calculation results; any number not found there is flagged as `unverified_numbers`.<br>
   **数字校验**：模型写出的每个数字都与计算结果比对，不在结果中的数字标记为 `unverified_numbers`

This addresses the risk in Section 8 of the Problem Statement: the model stating wrong numbers in a confident, professional tone.

这对应 Problem Statement 第 8 节的风险：模型用专业的语气说出错误的数字。

In [16]:
import json
import re

MODEL = "claude-haiku-4-5-20251001"
PRICE_INPUT_PER_M = 1.0    # USD per million input tokens (Haiku 4.5)
PRICE_OUTPUT_PER_M = 5.0   # USD per million output tokens (Haiku 4.5)

SYSTEM_PROMPT = """You are PauseBeforePay, a decision-support assistant that explains the financial consequences of three payment options for one purchase: pay in full, instalment, or delay the purchase.

Rules:
1. Use ONLY numbers that appear in the provided JSON. Never calculate, estimate or invent new numbers. If a number you want is not in the JSON, describe it in words instead.
2. Do not tell the user which option to choose. Explain the trade-offs clearly and let the user decide.
3. Write for a young adult with no finance background. Short sentences, no jargon; if you must use a term (for example "effective annual rate"), explain it in one short phrase.
4. Every option's main_risk must reflect its risk_flags. If an option has no flags, say its main risk is low.
5. Answer in {language}.
6. Return ONLY valid JSON, no markdown fences, no text before or after, in exactly this shape:
{{
  "overview": "2-3 sentences comparing the three options",
  "options": [
    {{"option": "pay_in_full", "plain_explanation": "...", "main_risk": "..."}},
    {{"option": "instalment", "plain_explanation": "...", "main_risk": "..."}},
    {{"option": "delay", "plain_explanation": "...", "main_risk": "..."}}
  ],
  "questions_to_ask_yourself": ["...", "..."]
}}"""


def build_payload(calc):
    """Send only what the model needs (no calculation steps) to save tokens.
    只发送模型需要的字段，不发运算步骤，节省 token。"""
    return {
        "currency": calc["currency"],
        "inputs": calc["inputs"],
        "settings": calc["settings"],
        "baseline": calc["baseline"],
        "options": calc["options"],
        "assumptions": calc["assumptions"],
    }


def parse_json(text):
    clean = re.sub(r"```(?:json)?", "", text).strip()
    return json.loads(clean)


REQUIRED_OPTIONS = {"pay_in_full", "instalment", "delay"}


def check_schema(out):
    problems = []
    for key in ["overview", "options", "questions_to_ask_yourself"]:
        if key not in out:
            problems.append(f"missing field: {key}")
    opts = out.get("options", [])
    names = {o.get("option") for o in opts if isinstance(o, dict)}
    if names != REQUIRED_OPTIONS:
        problems.append(f"options must be exactly {sorted(REQUIRED_OPTIONS)}, got {sorted(n for n in names if n)}")
    for o in opts:
        for k in ["plain_explanation", "main_risk"]:
            if not isinstance(o, dict) or not str(o.get(k, "")).strip():
                problems.append(f"option {o.get('option') if isinstance(o, dict) else o}: empty {k}")
    return problems


# ---------------------------------------------------------------- Number validator / 数字校验器
def _collect_numbers(obj, acc):
    if isinstance(obj, bool) or obj is None:
        return
    if isinstance(obj, (int, float)):
        acc.append(float(obj))
    elif isinstance(obj, dict):
        for v in obj.values():
            _collect_numbers(v, acc)
    elif isinstance(obj, (list, tuple)):
        for v in obj:
            _collect_numbers(v, acc)


def allowed_numbers(payload):
    raw = []
    _collect_numbers(payload, raw)
    allowed = set()
    for v in raw:
        for x in (v, v * 100, abs(v), abs(v) * 100):   # allow ratios as percentages and negatives as positive shortfalls / 比率可写成百分数，负数可写成缺口金额
            for d in (0, 1, 2):
                allowed.add(round(x, d))
    return allowed


NUM_RE = re.compile(r"(?<![A-Za-z])-?\d[\d,]*(?:\.\d+)?")


def validate_numbers(out, payload, tolerance=0.006):
    """Flag every number in the model output that is not in the calculation results.
    找出模型输出中不属于计算结果的数字。"""
    allowed = allowed_numbers(payload)
    text = json.dumps(out, ensure_ascii=False)
    unknown = []
    for m in NUM_RE.findall(text):
        try:
            x = float(m.replace(",", ""))
        except ValueError:
            continue
        if not any(abs(x - a) <= tolerance for a in allowed):
            unknown.append(m)
    return sorted(set(unknown))


def cost_usd(usage):
    return usage["input_tokens"] / 1e6 * PRICE_INPUT_PER_M + usage["output_tokens"] / 1e6 * PRICE_OUTPUT_PER_M


def explain(client, calc, language="English", max_tokens=1200):
    payload = build_payload(calc)
    resp = client.messages.create(
        model=MODEL,
        max_tokens=max_tokens,
        extra_body={"temperature": 0},   # SDK 1.x removed the temperature keyword ｜ SDK 1.x 移除了 temperature 参数，改用 extra_body 传入
        system=SYSTEM_PROMPT.format(language=language),
        messages=[{"role": "user", "content": json.dumps(payload, ensure_ascii=False)}],
    )
    text = "".join(b.text for b in resp.content if b.type == "text")
    usage = {"input_tokens": resp.usage.input_tokens, "output_tokens": resp.usage.output_tokens}
    result = {"raw_text": text, "usage": usage, "cost_usd": round(cost_usd(usage), 6)}
    try:
        out = parse_json(text)
    except json.JSONDecodeError as e:
        result.update({"ok": False, "error": f"invalid JSON: {e}"})
        return result
    schema_problems = check_schema(out)
    unknown = validate_numbers(out, payload)
    result.update({
        "ok": not schema_problems and not unknown,
        "explanation": out,
        "schema_problems": schema_problems,
        "unverified_numbers": unknown,
    })
    return result

First, test the validator offline (no API key needed): one correct output, and one that states a wrong instalment and quotes the misleading 7.2% rate.

先离线测试校验器（不需要 API key）：一段正确的输出，一段故意写错月供并引用表面利率 7.2% 的输出。

In [21]:
payload = build_payload(calc)
good = {"overview": "Paying in full leaves you SGD 1,000 short. Instalments cost SGD 536 a month and SGD 432 extra in total.",
        "options": [
            {"option": "pay_in_full", "plain_explanation": "You need 6,000 but have 5,000.", "main_risk": "Not enough savings."},
            {"option": "instalment", "plain_explanation": "12 payments of 536; your savings would last 1.65 months.", "main_risk": "Below your 3-month safety buffer."},
            {"option": "delay", "plain_explanation": "Save 1,000 a month for 6 months.", "main_risk": "Your buffer stays at 2 months."}],
        "questions_to_ask_yourself": ["Do I need it now?", "What if my income drops?"]}
bad = {**good, "overview": "Instalments cost SGD 520 a month at only 7.2% interest."}

print("good output -> unverified:", validate_numbers(good, payload), "| schema:", check_schema(good))
print("bad output  -> unverified:", validate_numbers(bad, payload))

good output -> unverified: [] | schema: []
bad output  -> unverified: ['520', '7.2']


Connect to the API (set `ANTHROPIC_API_KEY` in Secrets first).

连接 API（需要先在 Secrets 中设置 `ANTHROPIC_API_KEY`）。

In [32]:
# Connect to Claude Haiku 4.5 through OpenRouter ｜ 通过 OpenRouter 调用 Claude Haiku 4.5
!pip install -q openai
from openai import OpenAI
from types import SimpleNamespace as NS

try:
    from google.colab import userdata
    api_key = userdata.get("ANTHROPIC_API_KEY")   # your OpenRouter key is stored under this name ｜ 你的 OpenRouter key 存在这个名称下
except Exception:
    import getpass
    api_key = getpass.getpass("OpenRouter API key: ")

MODEL = "anthropic/claude-haiku-4.5"   # OpenRouter model ID ｜ OpenRouter 上的模型名称


class OpenRouterClaude:
    """Adapter: makes OpenRouter look like the Anthropic client, so the rest of the notebook stays unchanged.
    转接器：让 OpenRouter 的用法和 Anthropic 客户端一样，笔记本其他代码无需修改。"""
    def __init__(self, api_key):
        self._client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)
        self.messages = self

    def create(self, model, max_tokens, messages, system=None, extra_body=None, **kwargs):
        msgs = ([{"role": "system", "content": system}] if system else []) + messages
        temperature = (extra_body or {}).get("temperature", 0)
        r = self._client.chat.completions.create(model=model, max_tokens=max_tokens,
                                                 temperature=temperature, messages=msgs)
        text = r.choices[0].message.content or ""
        return NS(content=[NS(type="text", text=text)],
                  usage=NS(input_tokens=r.usage.prompt_tokens, output_tokens=r.usage.completion_tokens))


client = OpenRouterClaude(api_key.strip())
print("Client ready via OpenRouter, model =", MODEL)

Client ready via OpenRouter, model = anthropic/claude-haiku-4.5


In [33]:
result = explain(client, calc, language="English")   # change to "Chinese" for a Chinese explanation ｜ 改成 "Chinese" 可输出中文

print("Checks passed:", result["ok"])
print("Schema problems:", result.get("schema_problems"))
print("Unverified numbers:", result.get("unverified_numbers"))
print(f"Tokens: {result['usage']}  ->  cost per simulation: USD {result['cost_usd']:.6f}")
print()
print(json.dumps(result.get("explanation", result.get("raw_text")), indent=2, ensure_ascii=False))

Checks passed: True
Schema problems: []
Unverified numbers: []
Tokens: {'input_tokens': 986, 'output_tokens': 585}  ->  cost per simulation: USD 0.003911

{
  "overview": "Paying in full would drain your savings below a safe emergency cushion. Instalments let you keep your savings intact but cost you SGD 432 extra and reduce your monthly breathing room. Delaying 6 months lets you buy without extra costs, but you'll wait and your emergency fund stays thin either way.",
  "options": [
    {
      "option": "pay_in_full",
      "plain_explanation": "You'd pay SGD 6000 now with no extra fees. But your savings would drop to negative SGD 1000—meaning you'd have no safety net for emergencies like a medical bill or job loss. You'd need to rebuild from scratch.",
      "main_risk": "High. You'd have zero emergency savings, leaving you vulnerable to any unexpected expense."
    },
    {
      "option": "instalment",
      "plain_explanation": "You'd pay SGD 536 per month for 12 months, totalling

### Prompt v2 · Iteration ｜ 提示词 v2 · 迭代

The first real run passed all L1 checks but still contained errors that the number validator cannot catch: a number from one option reused for another (runway 2 months instead of 1.65), the monthly payment described as "extra", an unaffordable option described as payable, and the effective annual rate left out. Prompt v2 adds rules for each problem.

第一次真实运行通过了全部 L1 检查，但仍有数字校验器发现不了的错误：把一个方案的数字用到另一个方案上（应急金写成 2 个月，实际是 1.65 个月）、把月供说成"额外费用"、把付不起的方案说成可以支付、遗漏实际年化利率。v2 针对每个问题增加了一条规则。


In [34]:
 # ===== Prompt v2: fixes found in the first real run ｜ 提示词 v2：修正第一次真实运行中发现的问题 =====

SYSTEM_PROMPT_V2 = """You are PauseBeforePay, a decision-support assistant that explains the financial consequences of three payment options for one purchase: pay in full, instalment, or delay the purchase.

Rules:
1. Use ONLY numbers that appear in the provided JSON. Never calculate, estimate or invent new numbers. If a number you want is not in the JSON, describe it in words instead.
2. Every number must stay attached to its own option and meaning. When describing an option, use only the values inside options.<that option>. Use baseline values only when describing the situation today.
3. Emergency runway: for each option, quote that option's own emergency_runway_months. Never reuse another option's runway.
4. monthly_payment is the TOTAL amount repaid each month, not the extra cost. The extra cost is cost_of_credit, which is the total over the whole instalment period. Never describe monthly_payment as "extra".
5. If an option has feasible = false or the flag insufficient_savings, say clearly in the first sentence that the user cannot afford this option with current savings. Do not describe it as if the user could simply pay.
6. If the instalment option has cost_of_credit > 0, you MUST state its effective_annual_rate as a percentage (for example 0.1384 -> 13.84%) and explain in one sentence that the low per-month fee understates the true yearly cost.
7. Do not tell the user which option to choose. Explain the trade-offs and let the user decide.
8. Write for a young adult with no finance background. Short sentences, no jargon; explain any necessary term in one short phrase.
9. Every option's main_risk must reflect its risk_flags and start with its risk_level (High / Medium / Low).
10. Answer in {language}.
11. Return ONLY valid JSON, no markdown fences, no text before or after, in exactly this shape:
{{
  "overview": "2-3 sentences comparing the three options",
  "options": [
    {{"option": "pay_in_full", "plain_explanation": "...", "main_risk": "..."}},
    {{"option": "instalment", "plain_explanation": "...", "main_risk": "..."}},
    {{"option": "delay", "plain_explanation": "...", "main_risk": "..."}}
  ],
  "questions_to_ask_yourself": ["...", "..."]
}}"""


def explain_with_prompt(client, calc, system_prompt, language="English", max_tokens=1200):
    """Same as explain(), but lets you choose the system prompt, so v1 and v2 can be compared.
    与 explain() 相同，但可以指定提示词，便于对比 v1 和 v2。"""
    payload = build_payload(calc)
    resp = client.messages.create(
        model=MODEL,
        max_tokens=max_tokens,
        extra_body={"temperature": 0},
        system=system_prompt.format(language=language),
        messages=[{"role": "user", "content": json.dumps(payload, ensure_ascii=False)}],
    )
    text = "".join(b.text for b in resp.content if b.type == "text")
    usage = {"input_tokens": resp.usage.input_tokens, "output_tokens": resp.usage.output_tokens}
    result = {"raw_text": text, "usage": usage, "cost_usd": round(cost_usd(usage), 6)}
    try:
        out = parse_json(text)
    except json.JSONDecodeError as e:
        result.update({"ok": False, "error": f"invalid JSON: {e}"})
        return result
    schema_problems = check_schema(out)
    unknown = validate_numbers(out, payload)
    result.update({"ok": not schema_problems and not unknown, "explanation": out,
                   "schema_problems": schema_problems, "unverified_numbers": unknown})
    return result

In [35]:
result_v2 = explain_with_prompt(client, calc, SYSTEM_PROMPT_V2, language="English")

print("Checks passed:", result_v2["ok"])
print("Schema problems:", result_v2.get("schema_problems"))
print("Unverified numbers:", result_v2.get("unverified_numbers"))
print(f"Tokens: {result_v2['usage']}  ->  cost per simulation: USD {result_v2['cost_usd']:.6f}")
print()
print(json.dumps(result_v2.get("explanation", result_v2.get("raw_text")), indent=2, ensure_ascii=False))

Checks passed: False
Schema problems: []
Unverified numbers: ['36']
Tokens: {'input_tokens': 1205, 'output_tokens': 669}  ->  cost per simulation: USD 0.004550

{
  "overview": "Paying in full would leave you with no emergency savings, which is risky. Instalments let you keep your savings but cost SGD 432 extra and reduce your safety net. Delaying 6 months lets you buy without extra cost and keeps your emergency fund intact.",
  "options": [
    {
      "option": "pay_in_full",
      "plain_explanation": "You cannot afford this option with your current savings. Paying SGD 6000 now would leave you with negative savings (SGD -1000), meaning you'd have no money left for emergencies. Your monthly surplus stays at SGD 1000, but you'd have zero emergency runway instead of your current 2 months.",
      "main_risk": "High: You have insufficient savings to pay in full. This leaves you with no financial cushion if something unexpected happens, like a medical bill or job disruption. You'd be for

### Iteration Log · Prompt v1 → v2 ｜ 迭代记录 · 提示词 v1 → v2

Test case: income SGD 3,500, savings SGD 5,000, fixed outflow SGD 2,500, laptop SGD 6,000 in 12 instalments at 0.6% per instalment.

测试案例：月收入 3,500 新元，储蓄 5,000，每月固定支出 2,500，分 12 期购买 6,000 的电脑，每期手续费 0.6%。

| | v1 | v2 |
|---|---|---|
| L1 checks passed ｜ L1 检查通过 | ✅ Yes | ❌ No (1 unverified number) |
| Runway attached to the right option ｜ 应急金月数对应正确方案 | ❌ 2 months (should be 1.65) | ✅ 1.65 months |
| Monthly payment vs extra cost ｜ 月供与额外成本 | ❌ Called SGD 536 "extra" | ✅ SGD 536 monthly, SGD 432 extra |
| Unaffordable option stated clearly ｜ 明确指出付不起 | ❌ No | ✅ Yes, in the first sentence |
| Effective annual rate mentioned ｜ 提到实际年化利率 | ❌ No | ✅ 13.84% |
| Input / output tokens ｜ 输入 / 输出 token | 986 / 585 | 1,205 / 669 |
| Cost per simulation ｜ 每次模拟成本 | USD 0.0039 | USD 0.0046 |

**Finding 1 — L1 is necessary but not sufficient.** v1 passed every L1 check yet contained four semantic errors: each number existed in the calculation results but was attached to the wrong option or meaning. The number validator checks whether a number is traceable, not whether it is used correctly. This is why an L2 review (human or LLM judge) is needed.

**发现 1：L1 必要但不充分。** v1 通过了全部 L1 检查，却有 4 处语义错误：每个数字都存在于计算结果中，但被用在了错误的方案或含义上。数字校验器检查的是数字能否追溯来源，而不是数字是否用对了地方。所以需要 L2 评审（人工或 LLM 评审）。

**Finding 2 — the validator enforces traceability, not correctness.** v2 fixed all four errors, but to explain why the low fee is misleading, the model calculated the monthly fee itself (6,000 × 0.6% = SGD 36). The number is correct, yet it was flagged, because it did not come from the calculator. The fix is not to loosen the validator but to have the calculator output this value (`monthly_fee_amount`). If the explanation needs a number, Python must compute it.

**发现 2：校验器保证的是可追溯，而不是正确。** v2 修正了 4 处错误，但为了解释低费率为什么会误导人，模型自己算出了每月手续费（6,000 × 0.6% = 36 新元）。这个数字是对的，却仍被拦下，因为它不是计算器算出的。正确的修法不是放宽校验器，而是让计算器输出这个值（`monthly_fee_amount`）。解释需要哪个数字，就由 Python 来算。

**Finding 3 — quality has a cost.** The longer v2 prompt increased input tokens by about 22% and cost per simulation by about 16%, in exchange for fixing four errors that could mislead users.

**发现 3：质量提升有成本。** v2 的提示词更长，输入 token 增加约 22%，每次模拟成本增加约 16%，换来的是修正了 4 处可能误导用户的错误。

**Open issue for L2 review:** v2 says the user "would be forced to borrow immediately" under pay in full. This conclusion is not in the calculation results; it is a mild overreach rather than an error.

**留给 L2 评审的问题：** v2 在全款方案中写道用户"会被迫立刻借钱"。计算结果里没有这个结论，属于轻微的超出数据范围，而不是错误。


In [39]:
# ===== Round 3: prompt v2 + calculator now outputs the monthly fee ｜ 第三轮：v2 提示词 + 计算器输出每月手续费 =====

MODEL = "anthropic/claude-haiku-4.5"   # re-set in case an earlier cell reset it ｜ 防止前面的单元格把模型名称改回去

calc = simulate(profile, purchase, settings)   # recalculate with the updated calculator ｜ 用更新后的计算器重新计算
print("monthly_fee_amount from Python:", calc["options"]["instalment"]["monthly_fee_amount"])

result_v2b = explain_with_prompt(client, calc, SYSTEM_PROMPT_V2, language="English")

print("Checks passed:", result_v2b["ok"])
print("Schema problems:", result_v2b.get("schema_problems"))
print("Unverified numbers:", result_v2b.get("unverified_numbers"))
print(f"Tokens: {result_v2b['usage']}  ->  cost per simulation: USD {result_v2b['cost_usd']:.6f}")
print()
print(json.dumps(result_v2b.get("explanation", result_v2b.get("raw_text")), indent=2, ensure_ascii=False))

monthly_fee_amount from Python: 36.0
Checks passed: True
Schema problems: []
Unverified numbers: []
Tokens: {'input_tokens': 1217, 'output_tokens': 646}  ->  cost per simulation: USD 0.004447

{
  "overview": "Paying in full would drain your savings and leave you with no emergency buffer, so it's not possible right now. The instalment option lets you keep your savings intact but costs SGD 432 extra and cuts your monthly cushion in half. Delaying the purchase by 6 months lets you save up and buy it outright without any extra cost or monthly strain.",
  "options": [
    {
      "option": "pay_in_full",
      "plain_explanation": "You cannot afford this option with your current savings. Paying SGD 6000 now would leave you SGD 1000 short, with zero emergency savings left. You'd have no buffer if something unexpected happened.",
      "main_risk": "High: insufficient savings and no emergency fund. You would be financially exposed immediately."
    },
    {
      "option": "instalment",
    

## Part 5 · Try Your Own Case ｜ 第 5 部分 · 试你自己的案例

Edit the numbers and run. Use one fee type: `monthly_fee_rate` (flat fee per instalment, e.g. credit-card instalment plans) or `annual_interest_rate` (amortised loan). For 0% buy-now-pay-later, set both to 0.

修改数字后运行。费用参数二选一：`monthly_fee_rate`（按期手续费，如信用卡分期）或 `annual_interest_rate`（等额本息贷款）；0% 先买后付两个都填 0。

In [ ]:
my_profile = UserProfile(monthly_income=2500, cash_savings=2000,
                         essential_expenses=1800, existing_debt_payments=0)
my_purchase = Purchase(price=1600, instalment_months=6,
                       monthly_fee_rate=0.0, annual_interest_rate=0.0, upfront_fee=0)

my_calc = simulate(my_profile, my_purchase, Settings(currency="SGD"))
print_steps(my_calc)
if my_calc["ok"]:
    print_summary(my_calc)
    # uncomment the next line to get the model explanation ｜ 需要模型解释时取消下一行注释
    # r = explain(client, my_calc); print(json.dumps(r.get("explanation"), indent=2, ensure_ascii=False)); print("cost:", r["cost_usd"])

## Next Steps ｜ 下一步

- Move the code from Parts 1 and 4 into FastAPI as a `POST /simulate` endpoint for the Codex-built front end.

  把第 1、4 部分的代码搬进 FastAPI，提供 `POST /simulate` 接口给 Codex 做的前端调用。
- Run the 10-case test set through Part 4, recording `unverified_numbers` and `cost_usd` for the evaluation and cost sections of the report.

  用第 4 部分跑 10 个案例的测试集，记录 `unverified_numbers` 和 `cost_usd`，作为报告评估与成本部分的数据。